# 01 - Data Collection
Collecting TfL bike hire data, London weather and UK bank holidays, then merging them into one dataset.

In [2]:
%pip install requests openpyxl xlrd matplotlib

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
     ---------------------------------------- 0.0/47.3 kB ? eta -:--:--
     -------- ------------------------------- 10.2/47.3 kB ? eta -:--:--
     ------------------------ ------------- 30.7/47.3 kB 262.6 kB/s eta 0:00:01
     -------------------------------- ----- 41.0/47.3 kB 326.8 kB/s eta 0:00:01
     -------------------------------------- 47.3/47.3 kB 262.7 kB/s eta 0:00:00
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.11.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.wh


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import requests

In [6]:
path = "data/tfl-daily-cycle-hires-sep26-update.xlsx"

sheets = pd.read_excel(path, sheet_name=None)   # loads every sheet in the file
print("Sheet names:", list(sheets.keys()))

first = list(sheets.keys())[0]
sheets[first].head(10)

Sheet names: ['Metadata', 'Data']


,Name,Number of Bicycle Hires in the Santander Hire Scheme
0,ShortName,Number of Bicycle Hires
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,Theme,Transport
5,Sub-theme,Cycling
6,NaN,NaN
7,Title,Number of Bicycle Hires
8,Description,Total number of hires of the Santander Cycle H...
9,Subject,Cycling


In [7]:
raw = sheets["Data"]

print("Shape (rows, columns):", raw.shape)
print("\nColumn names:")
print(raw.columns.tolist())

raw.head(10)

Shape (rows, columns): (5912, 16)

Column names:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,NaN,Please note that from 01 August 2017 due to de...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Daily Grand Total,154920119,NaN,Monthly Grand Total,154920119,NaN,Annual Grand Total,154920119,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Day,Number of Bicycle Hires,NaN,Month,Number of Bicycle Hires,NaN,Year,Number of Bicycle Hires,NaN,NaN,Month,Average Hire Time (mins),NaN,Against last month,Against last year
5,NaN,2010-07-30 00:00:00,6897,NaN,2010-07-01 00:00:00,12461,NaN,2010,2180813,NaN,NaN,2010-07-01 00:00:00,17.232566,NaN,NaN,NaN
6,NaN,2010-07-31 00:00:00,5564,NaN,2010-08-01 00:00:00,341203,NaN,2011,7142449,NaN,NaN,2010-08-01 00:00:00,16.55188,NaN,NaN,NaN
7,NaN,2010-08-01 00:00:00,4303,NaN,2010-09-01 00:00:00,540859,NaN,2012,9519283,NaN,NaN,2010-09-01 00:00:00,15.219079,NaN,NaN,NaN
8,NaN,2010-08-02 00:00:00,6642,NaN,2010-10-01 00:00:00,544412,NaN,2013,8045459,NaN,NaN,2010-10-01 00:00:00,15.204481,NaN,NaN,NaN
9,NaN,2010-08-03 00:00:00,7966,NaN,2010-11-01 00:00:00,456304,NaN,2014,10023897,NaN,NaN,2010-11-01 00:00:00,13.776083,NaN,NaN,NaN


In [8]:
print(raw.iloc[0,1])

Please note that from 01 August 2017 due to delayed system events, daily hire figures are subject to small incremental changes. Figures should start to stabilise after a 2 week period.


In [9]:
# Keep only the daily columns (date + hires), skipping the header/notes rows
bikes = raw.iloc[5:, [1, 2]].copy()
bikes.columns = ["date", "hires"]

# Convert to the right data types
bikes["date"] = pd.to_datetime(bikes["date"], errors="coerce")
bikes["hires"] = pd.to_numeric(bikes["hires"], errors="coerce")

# Drop empty rows (the daily column is longer than the monthly one, so there may be blanks)
bikes = bikes.dropna().reset_index(drop=True)
bikes["hires"] = bikes["hires"].astype(int)

print(bikes.shape)
print("From", bikes["date"].min().date(), "to", bikes["date"].max().date())
bikes.head()

(5907, 2)
From 2010-07-30 to 2026-09-30


,date,hires
0,2010-07-30,6897
1,2010-07-31,5564
2,2010-08-01,4303
3,2010-08-02,6642
4,2010-08-03,7966


In [10]:
expected = pd.date_range(bikes["date"].min(), bikes["date"].max(), freq="D")
missing = expected.difference(bikes["date"])

print("Days expected:", len(expected))
print("Days in data: ", len(bikes))
print("Missing days: ", len(missing))
print("Duplicate dates:", bikes["date"].duplicated().sum())

Days expected: 5907
Days in data:  5907
Missing days:  0
Duplicate dates: 0


## 2. Get daily London weather
Historical daily weather for central London from the free Open-Meteo archive API.

In [11]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.5072,
    "longitude": -0.1276,
    "start_date": bikes["date"].min().strftime("%Y-%m-%d"),
    "end_date": bikes["date"].max().strftime("%Y-%m-%d"),
    "daily": "temperature_2m_mean,temperature_2m_max,precipitation_sum,wind_speed_10m_max,sunshine_duration",
    "timezone": "Europe/London",
}

response = requests.get(url, params=params, timeout=60)
print("Status:", response.status_code)   # 200 = success

weather = pd.DataFrame(response.json()["daily"])
weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temp_mean",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "rain_mm",
    "wind_speed_10m_max": "wind_max_kmh",
    "sunshine_duration": "sunshine_sec",
})
weather["date"] = pd.to_datetime(weather["date"])
weather["sunshine_hrs"] = (weather["sunshine_sec"] / 3600).round(1)
weather = weather.drop(columns="sunshine_sec")

print(weather.shape)
weather.head()

Status: 200
(5907, 6)


,date,temp_mean,temp_max,rain_mm,wind_max_kmh,sunshine_hrs
0,2010-07-30,17.7,22.0,0.0,16.5,11.9
1,2010-07-31,19.1,22.8,0.1,18.2,6.8
2,2010-08-01,17.7,21.8,0.0,11.9,9.8
3,2010-08-02,17.4,21.1,0.2,13.7,10.1
4,2010-08-03,17.6,21.7,0.0,17.7,9.1


## 3. Get UK bank holidays
Official England & Wales bank holiday dates from gov.uk.

In [12]:
bh = requests.get("https://www.gov.uk/bank-holidays.json", timeout=60).json()
bh_dates = pd.to_datetime([e["date"] for e in bh["england-and-wales"]["events"]])

print(len(bh_dates), "bank holidays")
print("Covers", bh_dates.min().date(), "to", bh_dates.max().date())

83 bank holidays
Covers 2019-01-01 to 2028-12-26


## 4. Merge datasets and create features
Join bike hires with weather on date, then add calendar features for analysis.

In [14]:
df = bikes.merge(weather, on="date", how="left")

# Calendar features
df["year"]        = df["date"].dt.year
df["month"]       = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()
df["is_weekend"]  = df["date"].dt.dayofweek >= 5      # Sat=5, Sun=6
df["is_bank_holiday"] = df["date"].isin(bh_dates)

df["season"] = df["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
})

# Weather flag
df["is_rainy"] = df["rain_mm"] >= 1.0                 # 1mm+ = a properly wet day

print(df.shape)
print("\nMissing values per column:")
print(df.isna().sum())
df.head()

(5907, 14)

Missing values per column:
date               0
hires              0
temp_mean          0
temp_max           0
rain_mm            0
wind_max_kmh       0
sunshine_hrs       0
year               0
month              0
day_of_week        0
is_weekend         0
is_bank_holiday    0
season             0
is_rainy           0
dtype: int64


,date,hires,temp_mean,temp_max,rain_mm,wind_max_kmh,sunshine_hrs,year,month,day_of_week,is_weekend,is_bank_holiday,season,is_rainy
0,2010-07-30,6897,17.7,22.0,0.0,16.5,11.9,2010,7,Friday,False,False,Summer,False
1,2010-07-31,5564,19.1,22.8,0.1,18.2,6.8,2010,7,Saturday,True,False,Summer,False
2,2010-08-01,4303,17.7,21.8,0.0,11.9,9.8,2010,8,Sunday,True,False,Summer,False
3,2010-08-02,6642,17.4,21.1,0.2,13.7,10.1,2010,8,Monday,False,False,Summer,False
4,2010-08-03,7966,17.6,21.7,0.0,17.7,9.1,2010,8,Tuesday,False,False,Summer,False


In [15]:
df.to_csv("data/bikes_weather_clean.csv", index=False)
print("Saved!")

Saved!
